# GDN2 v0.3.1 — Stage 2 + Stage 5(part) TPU evaluation
# Audit roadmap: gate audit on REAL activations (2.3), layer parity vs
# NVlabs reference (2.4), 0.3.1 kernel gates on TPU (Stage 5 part).
# Weights: Akseleu/gdn2-1.3b-hf (bit-exact conversion of the published
# 1.3B/95B GDN-2 checkpoint). Runtime: Colab TPU (v5e). Budget ~60-90 min.

In [ ]:
#@title 1. ENV: deps, HF download, inline v0.3.1 inference, fla self-test
import json, os, subprocess, sys, time, types, importlib.util, importlib.machinery
from getpass import getpass

T0 = time.time()
R = {"env": {}, "gates": {}, "parity": {}, "perf": {}}
os.makedirs("/content/results", exist_ok=True)

HF_TOKEN = getpass("HF token (read access to Akseleu/gdn2-1.3b-hf): ")
os.environ["HF_TOKEN"] = HF_TOKEN

if not os.path.exists("/content/atomic-ops"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/Akseleu-J/atomic-ops", "/content/atomic-ops"])
R["env"]["atomic_ops_head"] = subprocess.run(
    ["git", "-C", "/content/atomic-ops", "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip()

INFERENCE_V031 = r'''"""Inference kernels: single-token decode step and slim forward-only prefill.

Math follows reference/ref.py::token_serial_ref exactly (per token):
    Sbar = Diag(exp(min(g,0))) S
    erase = (b*k) @ Sbar                       # (b*k) . Sbar, per value channel
    v_new = w*v - erase
    S = Sbar + k (v_new)^T
    o = S^T (scale*q)

Deltas vs the training path (fwd/cd_slim.py): inference keeps only (o, final
state) — no h_pre_all / v_new_all / gc_last residuals are materialized — and
adds the serving features documented per function: per-row reset, segment
(document) resets at chunk boundaries, bfloat16 state storage, and an optional
Frobenius-norm stability cap on the returned state. Mid-chunk document
boundaries are not supported at this level: packed batches should split so a
chunk never spans two documents (the vLLM layer owns that scheduling).

All kernels honor cfg.interpret (CPU emulation) and run on TPU otherwise.
"""
import numpy as np

import jax, jax.numpy as jnp
from jax.experimental import pallas as pl
from .precision import exp_nonpos, sanitize
from .layout import cparams
from .config import BLRConfig

__all__ = ["decode_step", "chunk_forward_inference", "LayerInputs"]

import dataclasses as _dc


@_dc.dataclass(frozen=True)
class LayerInputs:
    """Named input container (audit Stage 3.1, fixes B3 arg-order confusion).

    All tensors are head-ready: q/k (B, H, Dk); v/w (B, Hv, Dv); g (B, H, Dk)
    per-token log-decay (<= 0); b (B, H, Dk). Both entry points accept this
    container instead of six positional tensors, making argument order
    impossible to get wrong.
    """
    q: jax.Array
    k: jax.Array
    v: jax.Array
    w: jax.Array
    b: jax.Array
    g: jax.Array

    def validate(self, Dk: int, Dv: int):
        B = self.q.shape[0]
        want = {"q": (B, self.q.shape[1], Dk), "k": (B, self.q.shape[1], Dk),
                "v": (B, self.v.shape[1], Dv), "w": (B, self.v.shape[1], Dv),
                "b": (B, self.q.shape[1], Dk), "g": (B, self.q.shape[1], Dk)}
        for name, arr in (("q", self.q), ("k", self.k), ("v", self.v),
                          ("w", self.w), ("b", self.b), ("g", self.g)):
            if tuple(arr.shape) != want[name]:
                raise ValueError(f"{name}: expected {want[name]}, got {tuple(arr.shape)}")

    def as_tuple(self):
        return (self.q, self.k, self.v, self.w, self.b, self.g)

PAD_N = 128  # segment-block padding for TPU Mosaic block-shape legality


def _decode_body(state_ref, q_ref, k_ref, v_ref, w_ref, b_ref, g_ref,
                 a_log_ref, dt_bias_ref, o_ref, new_ref, *,
                 scale: float, stability_tau, fuse_prologue: bool = False):
    S = state_ref[0].astype(jnp.float32)
    q = q_ref[0].astype(jnp.float32)
    k = k_ref[0].astype(jnp.float32)
    v = v_ref[0].astype(jnp.float32)
    w = w_ref[0].astype(jnp.float32)
    b = b_ref[0].astype(jnp.float32)
    g = g_ref[0].astype(jnp.float32)
    if fuse_prologue:
        # activations fused into the kernel (prologue-fusion method adapted
        # from the deltanet-tpu inference stack) using the CHECKPOINT gate
        # construction: g = -exp(A_log) * softplus(f_proj(x) + dt_bias),
        # b/w are sigmoid logits, q/k are l2-normalized in-kernel.
        q = q * jax.lax.rsqrt(jnp.sum(q * q, -1, keepdims=True) + 1e-6)
        k = k * jax.lax.rsqrt(jnp.sum(k * k, -1, keepdims=True) + 1e-6)
        b = jax.nn.sigmoid(b)
        w = jax.nn.sigmoid(w)
        a_log = a_log_ref[0].astype(jnp.float32)      # (H,)
        dt = dt_bias_ref[0].astype(jnp.float32)       # (K,)
        g = -jnp.exp(a_log)[..., None] * jax.nn.softplus(g + dt[None, :])
    h = S * jnp.exp(jnp.minimum(g, 0.0))[..., None]
    erase = jnp.sum((b * k)[..., None] * h, axis=1)
    v_new = w * v - erase
    h = h + k[..., None] * v_new[..., None, :]
    o = jnp.sum(h * (q * scale)[..., None], axis=1)
    if stability_tau is not None:
        # per-head Frobenius norm (B1): heads are independent, one head must
        # not suppress the others; matches the prefill cap semantics.
        nrm = jnp.sqrt(jnp.sum(h * h, axis=(-2, -1)))      # (H,)
        h = h * jnp.minimum(1.0, stability_tau / jnp.maximum(nrm, 1e-12))[..., None, None]
    # A5 policy (owner decision): a non-finite entry never persists — the
    # in-kernel cleanup replaces NaN/Inf state and output entries with zeros
    # (the affected row effectively restarts from a clean state). Done
    # IN-KERNEL: post-pallas jnp.where on interpret-mode outputs trips ref
    # indexing semantics.
    h = jnp.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
    o = jnp.nan_to_num(o, nan=0.0, posinf=0.0, neginf=0.0)
    o_ref[0] = o.astype(o_ref.dtype)
    new_ref[0] = h.astype(new_ref.dtype)


def decode_step(q=None, k=None, v=None, w=None, b=None, g=None, state=None,
                scale: float = 1.0, reset=None, state_dtype=jnp.float32,
                stability_tau=None, fuse_prologue: bool = False, a_log=None,
                dt_bias=None, inputs: "LayerInputs | None" = None,
                cfg: BLRConfig | None = None):
    """Single-token recurrence for serving.

    q, k, v, w, b, g: (B, H, D); g is THIS token's log-decay (g <= 0 by the
    layer's domain construction; positive values are clamped to 0 with the
    same semantics as token_serial_ref). state: (B, H, D, D) in state_dtype.
    reset: (B,) int/bool — zero the incoming state for those rows (vLLM
    preemption / document boundaries).

    fuse_prologue: when True, q/k are RAW (l2-normalized in-kernel), b/w are
    raw gate logits (sigmoid in-kernel) and g is the pre-softplus projection
    (g = -min(softplus(g), g_max) in-kernel) — activations never materialize
    in HBM (deltanet-tpu's prologue-fusion method adapted to our domain
    contract). Requires cfg.

    Returns (o (B, H, D), new_state (B, H, D, D) in state_dtype). Compute is
    float32; storage dtype is state_dtype ("bfloat16" halves state bytes; the
    loss is a single rounding of the returned state, validated for the
    channel-wise GDN-2 update).
    """
    if inputs is not None:
        q, k, v, w, b, g = inputs.as_tuple()
    if q is None or state is None:
        raise ValueError("pass either inputs=LayerInputs(...) or q,k,v,w,b,g,state")
    bsz, H, D = q.shape
    if not isinstance(scale, (int, float)) or isinstance(scale, jax.Array):
        raise ValueError(
            "scale must be a concrete Python float (B4): traced values cannot "
            "enter the Pallas kernel; pass it as a compile-time constant")
    if stability_tau is not None and not isinstance(stability_tau, (int, float)):
        raise ValueError(
            "stability_tau must be a concrete Python float (B4)")
    if state.shape != (bsz, H, D, D):
        raise ValueError(f"state must be (B,H,D,D), got {state.shape}")
    st = state.astype(state_dtype)
    if reset is not None:
        # per-row reset happens OUTSIDE the kernel: a (B,) scalar block would
        # violate the TPU Mosaic block-shape rule (the last two dims of a block
        # must divide (8, 128) or equal the array's dims).
        rst = jnp.asarray(reset).reshape(bsz).astype(jnp.int32)
        st = jnp.where(rst[:, None, None, None] > 0, jnp.zeros_like(st), st)
    if fuse_prologue:
        if a_log is None or dt_bias is None:
            raise ValueError(
                "fuse_prologue=True requires a_log (H,) and dt_bias (K,) from "
                "the checkpoint: the kernel then computes "
                "g = -exp(A_log) * softplus(g_raw + dt_bias) in-kernel")
        a_log = jnp.asarray(a_log).reshape(H)
        dt_bias = jnp.asarray(dt_bias).reshape(D)
    # grid (B,); blocks span the FULL (H, .) extent of their last two axes, so
    # they are legal on TPU Mosaic for any H, D (block dims == array dims).
    grid = (bsz,)
    ss = pl.BlockSpec((1, H, D, D), lambda i: (i, 0, 0, 0))
    vs = pl.BlockSpec((1, H, D), lambda i: (i, 0, 0))
    # gate params as (1, X) full-array 2-D blocks (legal: block == array dims);
    # body reads them with the same ref[0] pattern as every other ref
    als = pl.BlockSpec((1, H), lambda i: (0, 0))
    dts = pl.BlockSpec((1, D), lambda i: (0, 0))
    a_log_in = (jnp.asarray(a_log).reshape(1, H) if fuse_prologue
                else jnp.zeros((1, H), jnp.float32))
    dt_in = (jnp.asarray(dt_bias).reshape(1, D) if fuse_prologue
             else jnp.zeros((1, D), jnp.float32))
    o, new = pl.pallas_call(
        lambda *r: _decode_body(*r, scale=float(scale), stability_tau=stability_tau,
                                fuse_prologue=fuse_prologue),
        grid=grid,
        in_specs=[ss, vs, vs, vs, vs, vs, vs, als, dts],
        out_specs=[vs, ss],
        out_shape=[jax.ShapeDtypeStruct((bsz, H, D), q.dtype),
                   jax.ShapeDtypeStruct((bsz, H, D, D), state_dtype)],
        compiler_params=cparams(cfg.vmem_mb if cfg is not None else 150.0),
        interpret=(cfg.interpret if cfg is not None else jax.default_backend() != "tpu"),
    )(st, q, k, v, w, b, g, a_log_in, dt_in)
    # materialize: in interpret mode the outputs are Ref objects whose fancy
    # indexing follows ref semantics (leading-axis unwrap), which breaks the
    # guard below — round-trip through numpy to get plain jnp arrays (no-op
    # on TPU where the outputs are real arrays already).
    o = jnp.asarray(np.asarray(o))
    new = jnp.asarray(np.asarray(new))
    return o, new


def _cd_inference_body(q_ref, k_ref, v_ref, w_ref, b_ref, gc_ref, a_ref, aqk_ref,
                       h0_ref, seg_ref, o_ref, hf_ref, *, nc: int, scale: float,
                       cfg: BLRConfig, seg_mode: bool):
    from .precision import make_dot
    dot = make_dot(cfg.dot_mode)
    h = h0_ref[0, 0].astype(jnp.float32)
    segs = seg_ref[0][0] if seg_mode else None  # row 0 of the (8, PAD_N) block = this batch row
    for c in range(nc):
        if seg_mode and c > 0:
            h = jnp.where(segs[c] != segs[c - 1], jnp.zeros_like(h), h)
        q = q_ref[0, 0, c].astype(jnp.float32)
        k = k_ref[0, 0, c].astype(jnp.float32)
        v = v_ref[0, 0, c].astype(jnp.float32)
        w = w_ref[0, 0, c].astype(jnp.float32)
        b = b_ref[0, 0, c].astype(jnp.float32)
        gc = gc_ref[0, 0, c].astype(jnp.float32)
        A = a_ref[0, 0, c].astype(jnp.float32)
        Aq = aqk_ref[0, 0, c].astype(jnp.float32)
        egc, _ = exp_nonpos(gc)
        wp = sanitize(dot(A, b * k * egc), cfg.clip)
        u = sanitize(dot(A, w * v), cfg.clip)
        gc_last = gc[cfg.bt - 1]
        ekg, _ = exp_nonpos(gc_last[None, :] - gc)
        kg = sanitize(k * ekg, cfg.clip)
        qg = sanitize(q * egc, cfg.clip)
        v_new = u - dot(wp, h)
        o_c = scale * dot(qg, h) + dot(Aq, v_new)
        dec, _ = exp_nonpos(gc_last)
        h = sanitize(h * dec[:, None] + dot(kg.T, v_new), cfg.clip)
        o_ref[0, 0, c] = sanitize(o_c, cfg.clip)
    hf_ref[0, 0] = h


def chunk_forward_inference(q=None, k=None, v=None, w=None, b=None, g=None,
                            scale: float = 1.0, cfg: BLRConfig | None = None,
                            h0=None, segment_ids=None, cu_seqlens=None,
                            state_dtype=jnp.float32, stability_tau=None,
                            inputs: "LayerInputs | None" = None):
    """Forward-only prefill with bounded outputs: returns (o, h_final) only.

    Same math and layouts as pipeline.make_blr_trainable's forward (head-major
    (B,H,L,D), L % cfg.bt == 0, segments of cfg.max_chunks_per_call chunks with
    the state carried between them), but no training residuals are produced and
    the recurrent state is returned in state_dtype.

    segment_ids: optional (B, L) int tensor of document IDs. A document
    boundary resets the state; boundaries are honored at CHUNK granularity in
    this level (a chunk spanning two documents must be split by the caller —
    the vLLM layer owns that scheduling). cu_seqlens: optional 1-D token
    offsets; converted to per-token segment ids internally (same chunk
    granularity caveat). initial_state is assumed to belong
    to the first chunk's document.
    stability_tau: optional cap on the Frobenius norm of the returned state
    (inference-only; the training path is never capped).
    """
    if inputs is not None:
        q, k, v, w, b, g = inputs.as_tuple()
    bsz, H, L, D = q.shape
    # B7 fix (0.4.0): arbitrary length — full chunks go through the chunkwise
    # kernel, the tail (L % bt tokens) goes through sequential decode steps.
    # The old strict `L % bt == 0` requirement is gone; the vLLM layer can
    # still schedule chunk-aligned sequences for maximum throughput.
    # B2 fix: decode accepts bfloat16 inputs, prefill crashed on ref dtype
    # swap — normalize to float32 at the API boundary (compute is fp32 anyway)
    q, k, v, w, b = (jnp.asarray(t).astype(jnp.float32) for t in (q, k, v, w, b))
    n_full = L // cfg.bt
    r_tail = L - n_full * cfg.bt
    scale = float(scale)
    # A6 fix (audit): prefill never validated the g domain — g=-8 (beyond
    # g_max) gave 58% and g>0 gave 205% error silently. Clamp identically to
    # the decode path (g <= 0) and raise loudly when outside the layer's
    # documented domain [-g_max, 0].
    g = jnp.minimum(jnp.asarray(g).astype(jnp.float32), 0.0)
    g_max_abs = float(cfg.g_max)
    try:
        g_np = np.asarray(g)                     # concrete only; skipped under jit
        g_bad = bool((g_np < -g_max_abs - 1e-6).any())
    except (TypeError, ValueError):
        g_bad = False
    if g_bad:
        raise ValueError(
            f"g outside the layer domain: min(g) < -g_max={g_max_abs} "
            "(the layer constructs g = -min(softplus(...), g_max); clamp or "
            "fix the projection before calling)")
    h = jnp.zeros((bsz, H, D, D), jnp.float32) if h0 is None else h0.astype(jnp.float32)

    n_full = L // cfg.bt
    L_full = n_full * cfg.bt
    r_tail = L - L_full
    if segment_ids is not None and n_full > 0:
        seg_tok = jnp.asarray(segment_ids).reshape(bsz, L)[:, :L_full]
        # A2 fix (owner decision): a document boundary INSIDE a chunk used to
        # be silently ignored (20.9% error). Raise loudly instead — the
        # caller must split chunks at document boundaries. The TAIL (after
        # the full chunks) is exempt: decode steps handle per-token resets.
        per_chunk_first = seg_tok[:, ::cfg.bt]                       # (B, n_full)
        same = (seg_tok.reshape(bsz, n_full, cfg.bt)
                == per_chunk_first[:, :, None])                      # (B, n_full, bt)
        bad = ~(same.all(axis=-1))                                   # (B, n_full)
        try:
            bad_concrete = bool(np.asarray(bad.any()))
        except (TypeError, ValueError):
            bad_concrete = False
        if bad_concrete:
            b_idx, c_idx = np.argwhere(np.asarray(bad))[0]
            raise ValueError(
                f"document boundary inside chunk {int(c_idx)} of batch row "
                f"{int(b_idx)} (tokens start at {int(c_idx) * cfg.bt}): split "
                "the sequence at document boundaries before calling "
                "chunk_forward_inference (chunk-granularity resets only)")
    if cu_seqlens is not None:
        # A3 fix: any number of documents, any batch size. cu_seqlens is either
        # 1-D (shared by all rows) or (B, n_docs+1). A boundary token STARTS a
        # new document (audit: the earlier `>` semantics shifted resets late).
        if segment_ids is not None:
            raise ValueError("pass either segment_ids or cu_seqlens, not both")
        cs = jnp.asarray(cu_seqlens)
        if cs.ndim == 1:
            cs = jnp.broadcast_to(cs[None, :], (bsz, cs.shape[0]))
        n_int = cs.shape[1] - 2
        pos = jnp.arange(L)
        if n_int <= 0:
            segment_ids = jnp.zeros((bsz, L), jnp.int32)   # single document
        else:
            bounds = cs[:, 1:-1]                           # (B, n_int)
            # a boundary token STARTS its document
            segment_ids = (pos[None, :, None] >= bounds[:, None, :]).sum(axis=-1).astype(jnp.int32)
    seg_chunk = None
    seg_padded = None
    if segment_ids is not None:
        seg = jnp.asarray(segment_ids).reshape(bsz, L)
        seg_chunk = seg[:, :L_full:cfg.bt][:, :n_full]  # (B, n_full) seg at chunk start
        # A1 fix: pad to (B, 8, PAD_N) so each program can read ITS row's
        # boundaries through a TPU-legal block (row 0 = this row, 1-7 pad)
        seg_padded = jnp.zeros((bsz, 8, PAD_N), dtype=seg_chunk.dtype)
        seg_padded = seg_padded.at[:, 0, :n_full].set(seg_chunk)

    outs = []
    for s in range(0, n_full, cfg.max_chunks_per_call):
        e = min(n_full, s + cfg.max_chunks_per_call)
        n = e - s
        sl = lambda t: t[:, :, s * cfg.bt:e * cfg.bt].reshape(bsz, H, n, cfg.bt, D)
        q_c, k_c, v_c, w_c, b_c = sl(q), sl(k), sl(v), sl(w), sl(b)
        # gc is the PER-CHUNK cumulative decay (pipeline computes it on the
        # chunked tensor, axis=-2 == the bt axis), not over the whole sequence.
        gc_c = jnp.cumsum(sl(g).astype(jnp.float32), axis=-2)
        if seg_chunk is not None:
            seg_cur = seg_chunk[:, s:e]
            if s > 0:
                # document changed at the segment boundary -> zero those rows
                boundary = seg_cur[:, 0] != seg_chunk[:, s - 1]
                h = jnp.where(boundary[:, None, None, None], 0.0, h)
            h_in = h
        else:
            seg_cur = None
            h_in = h
        Aqk, Akk = build_scores_local(q_c, k_c, b_c, gc_c, scale, cfg)
        A = solve_local(Akk, cfg)
        io = pl.BlockSpec((1, 1, n, cfg.bt, D), lambda i, hh: (i, hh, 0, 0, 0))
        sc = pl.BlockSpec((1, 1, n, cfg.bt, cfg.bt), lambda i, hh: (i, hh, 0, 0, 0))
        hs = pl.BlockSpec((1, 1, D, D), lambda i, hh: (i, hh, 0, 0))
        # full-array block: legal on TPU Mosaic for any (B, n) shape
        # A1 fix: per-ROW segment blocks. A (B, n) full-array block forced
        # every program to read row 0's boundaries (audit blocker A1: with
        # B>1 all rows used row 0's document layout, silently mixing states).
        # TPU Mosaic legality: the block must be (1, 8, 128) — axes padded to
        # (8, 128) so the divisibility rule holds; row 0 of the padded block
        # is this program's batch row, rows 1-7 are zero padding.
        sg = pl.BlockSpec((1, 8, PAD_N), lambda i, hh: (i, 0, 0))
        o_ch, hf = pl.pallas_call(
            lambda *r: _cd_inference_body(*r, nc=n, scale=scale, cfg=cfg,
                                          seg_mode=seg_cur is not None),
            grid=(bsz, H),
            in_specs=[io, io, io, io, io, io, sc, sc, hs, sg],
            out_specs=[io, hs],
            out_shape=[jax.ShapeDtypeStruct((bsz, H, n, cfg.bt, D), q.dtype),
                       jax.ShapeDtypeStruct((bsz, H, D, D), jnp.float32)],
            compiler_params=cparams(cfg.vmem_mb),
            interpret=cfg.interpret,
        )(q_c, k_c, v_c, w_c, b_c, gc_c, A, Aqk, h_in.astype(jnp.float32),
          seg_padded if seg_padded is not None else jnp.zeros((bsz, 8, PAD_N), jnp.int32))
        # materialize interpret-mode Refs to plain arrays (see decode_step)
        h = jnp.asarray(np.asarray(hf))
        outs.append(from_chunks_local(jnp.asarray(np.asarray(o_ch))))
    o = (jnp.concatenate(outs, axis=2) if len(outs) > 1
         else (outs[0] if outs else None))

    # ---- B7: decode the tail (L % bt tokens) with per-token doc resets ----
    if r_tail > 0:
        tail_resets = None
        if seg_chunk is not None:
            seg_full = jnp.asarray(segment_ids).reshape(bsz, L)
            if L_full > 0:
                prev = jnp.concatenate([seg_full[:, L_full - 1:L_full],
                                        seg_full[:, L_full:-1]], axis=1)[:, :r_tail]
            else:
                prev = jnp.zeros((bsz, r_tail), jnp.int32)
            tail_resets = (seg_full[:, L_full:] != prev).astype(jnp.int32)  # (B, r)
        outs_tail = []
        for j in range(r_tail):
            t_abs = L_full + j
            rst = (tail_resets[:, j] if tail_resets is not None else None)
            o_t, h = decode_step(q[:, :, t_abs], k[:, :, t_abs], v[:, :, t_abs],
                                 w[:, :, t_abs], b[:, :, t_abs], g[:, :, t_abs],
                                 h.astype(state_dtype), scale, reset=rst,
                                 state_dtype=state_dtype, cfg=cfg)
            outs_tail.append(o_t)
        o_tail = jnp.stack(outs_tail, axis=2)                    # (B, H, r, D)
        o = (o_tail if o is None else jnp.concatenate([o, o_tail], axis=2))

    if stability_tau is not None:
        nrm = jnp.sqrt(jnp.sum(h.astype(jnp.float32) ** 2, axis=(-2, -1), keepdims=True))
        h = h * jnp.minimum(1.0, stability_tau / jnp.maximum(nrm, 1e-12))
    return o, h.astype(state_dtype)


def build_scores_local(q, k, b, gc, scale, cfg):
    from .fwd import build_scores
    return build_scores(q, k, b, gc, scale, cfg)


def solve_local(Akk, cfg):
    from .fwd import wy_solve_leafbatched
    return wy_solve_leafbatched(Akk, cfg)


def from_chunks_local(t):
    b, h, nc, bt, d = t.shape
    return t.reshape(b, h, nc * bt, d)
'''
pkg = "/content/atomic-ops/atomic_gdn2"
with open(os.path.join(pkg, "inference.py"), "w", encoding="utf-8") as fh:
    fh.write(INFERENCE_V031)
sys.path.insert(0, "/content/atomic-ops")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "safetensors", "einops", "sentencepiece", "huggingface_hub",
                    "flash-linear-attention"],
                   capture_output=True, text=True)
print("pip exit:", r.returncode, r.stderr[-200:] if r.returncode else "")
r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "triton"],
                    capture_output=True, text=True)
print("triton upgrade exit:", r2.returncode)

# Triton/FLA compat: FLA's CachedAutotuner reads self.user_defined_pre_hook,
# set only by newer Triton (triton-lang/triton#10295). Pre-import default:
import triton
from triton.runtime import autotuner as _ta
if not hasattr(_ta.Autotuner, "user_defined_pre_hook"):
    _ta.Autotuner.user_defined_pre_hook = None
if not hasattr(_ta.Autotuner, "user_defined_post_hook"):
    _ta.Autotuner.user_defined_post_hook = None
print("triton:", triton.__version__, "| Autotuner hooks patched if missing")

# flash_attn stub: dense model has no attention layers. __spec__/__path__ are
# REQUIRED — fla's import machinery calls find_spec("flash_attn") and raises
# ValueError when __spec__ is None (round-1 failure).
fa = types.ModuleType("flash_attn")
fa.flash_attn_func = fa.flash_attn_varlen_func = None
fa.__path__ = []
fa.__spec__ = importlib.machinery.ModuleSpec("flash_attn", None, is_package=True)
bp = types.ModuleType("flash_attn.bert_padding")
bp.index_first_axis = bp.pad_input = bp.unpad_input = None
bp.__spec__ = importlib.machinery.ModuleSpec("flash_attn.bert_padding", None)
fa.bert_padding = bp
sys.modules["flash_attn"] = fa
sys.modules["flash_attn.bert_padding"] = bp

import jax
import torch
R["env"]["jax"] = jax.__version__
R["env"]["torch"] = torch.__version__
R["env"]["jax_devices"] = [str(d) for d in jax.devices()]
print("jax", jax.__version__, "| devices:", R["env"]["jax_devices"])
print("torch", torch.__version__)

from huggingface_hub import hf_hub_download
wt = hf_hub_download(repo_id="Akseleu/gdn2-1.3b-hf", filename="model.safetensors",
                     repo_type="model", token=HF_TOKEN, cache_dir="/content/hf")
R["env"]["weights"] = wt
print("weights:", wt)

tok_dir = "/content/tok"; os.makedirs(tok_dir, exist_ok=True)
for f in ("tokenizer.model", "tokenizer.json", "tokenizer_config.json"):
    if not os.path.exists(os.path.join(tok_dir, f)):
        hf_hub_download(repo_id="Akseleu/gdn2-1.3b-hf", filename=f,
                        repo_type="model", local_dir=tok_dir, token=HF_TOKEN)
from sentencepiece import SentencePieceProcessor
sp = SentencePieceProcessor(model_file=os.path.join(tok_dir, "tokenizer.model"))
R["env"]["tokenizer_vocab"] = sp.get_piece_size()
def sp_encode_fn(text):
    ids = sp.encode(text)
    while len(ids) < 128:
        ids = ids + sp.encode(text)
    return ids[:128]
print("tokenizer vocab:", sp.get_piece_size())

# fla self-test — the exact import that crashed on old Triton; must be green
import fla
from fla.modules import FusedRMSNormSwishGate, ShortConvolution
_g = FusedRMSNormSwishGate(128)(torch.randn(1, 4, 128), torch.randn(1, 4, 128))
print("fla self-test OK:", tuple(_g.shape))

with open("/content/results/env_manifest.json", "w") as fh:
    json.dump(R["env"], fh, indent=2)
print("ENV MANIFEST WRITTEN")

In [ ]:
#@title 2. [RUN 1] load weights + NVlabs model (CPU fp32), capture layer I/O
import numpy as np
import glob, re
from safetensors.torch import load_file
sys.path.insert(0, "/content/GatedDeltaNet-2")

cands = glob.glob("/content/hf/**/*.safetensors", recursive=True)
assert cands, "safetensors not found in hf cache"
sd = load_file(cands[0])
print("safetensors:", len(sd), "tensors")

from lit_gpt.config import Config
from lit_gpt.model import GPT
cfg = Config(org="NVIDIA", name="gdn2_1.3B")
cfg.padded_vocab_size = cfg.vocab_size

def to_lit(sd):
    out = {}
    for k, v in sd.items():
        if k == "lm_head.weight": out["lm_head.weight"] = v; continue
        if k == "model.embed_tokens.weight": out["transformer.wte.weight"] = v; continue
        if k == "model.norm.weight": out["transformer.ln_f.weight"] = v; continue
        m = re.fullmatch(r"model\.layers\.(\d+)\.(.+)", k)
        i, rest = m.groups()
        rest = rest.replace("linear_attn.", "attn.") \
                   .replace("mlp.gate_proj.", "mlp.swiglu.w1.") \
                   .replace("mlp.up_proj.", "mlp.swiglu.w2.") \
                   .replace("mlp.down_proj.", "mlp.swiglu.w3.") \
                   .replace("input_layernorm.", "norm_1.") \
                   .replace("post_attention_layernorm.", "norm_2.") \
                   .replace("_conv1d.conv.", "_conv1d.")
        out[f"transformer.h.{i}.{rest}"] = v
    return out

model = GPT(cfg)
res = model.load_state_dict(to_lit(sd), strict=False)
assert not res.unexpected_keys, res.unexpected_keys[:5]
print("missing keys:", res.missing_keys[:5])
model = model.eval()

prompt = sp_encode_fn("The purpose of linear attention is to replace softmax "
                      "attention with a recurrent state update, so that "
                      "generation cost stays constant per token. Gated DeltaNet "
                      "2 decouples the erase and write gates, giving the model "
                      "independent control over forgetting and writing in "
                      "every channel of the recurrent state.")
ids = torch.tensor([prompt], dtype=torch.long)
print("prompt tokens:", len(prompt))

captured = {}
def mk_hooks(idx, attn_module):
    def pre(module, args, kwargs):
        captured.setdefault(idx, {})["x"] = args[0].detach().clone()
    def post(module, args, kwargs, out):
        captured.setdefault(idx, {})["o"] = (out[0] if isinstance(out, tuple) else out).detach().clone()
    attn_module.register_forward_pre_hook(pre)
    attn_module.register_forward_hook(post, with_kwargs=True)

for i, block in enumerate(model.transformer.h):
    mk_hooks(i, block.attn)

with torch.no_grad():
    logits = model(ids)
print("logits:", tuple(logits.shape))
R["env"]["prompt_tokens"] = len(prompt)

In [ ]:
#@title 3. [RUN 1] gate audit (Stage 2.3): per-layer |g| + BTL span validity
import numpy as np

def recompute_g(layer_idx, x):
    # exactly lit_gpt GatedDeltaNet2.forward:
    # g = -A_log.exp().repeat_interleave(head_k_dim) * softplus(f_proj(x)+dt_bias)
    blk = model.transformer.h[layer_idx].attn
    f = blk.f_proj(x.float())
    a_log = blk.A_log.float()
    dt = blk.dt_bias.float()
    return -a_log.exp().repeat_interleave(blk.head_k_dim) * torch.nn.functional.softplus(f + dt)

LAYERS = sorted(captured.keys())
gate_stats = {}
for i in LAYERS:
    x_in = captured[i]["x"]
    g = recompute_g(i, x_in)[0].numpy()             # (L, K) fp32
    bt_, bs2_ = 128, 32
    nc_ = g.shape[0] // bt_
    max_half_span = 0.0
    for c in range(nc_):
        gc = np.cumsum(g[c*bt_:(c+1)*bt_], axis=0)
        for bb in range(bt_ // bs2_):
            blk = gc[bb*bs2_:(bb+1)*bs2_]
            span = np.abs(blk[:, None, :] - blk[None, :, :]).max()
            max_half_span = max(max_half_span, float(span))
    gate_stats[i] = {
        "abs_g_max": float(np.abs(g).max()),
        "abs_g_mean": float(np.abs(g).mean()),
        "abs_g_p99": float(np.percentile(np.abs(g), 99)),
        "frac_abs_g_gt_5": float((np.abs(g) > 5).mean()),
        "max_half_span_observed": max_half_span,
        "btl_limit_88": bool(max_half_span < 88.0),
    }
    st = gate_stats[i]
    print(f"layer {i:2d}: |g|max={st['abs_g_max']:.3f} mean={st['abs_g_mean']:.4f} "
          f"frac>5={st['frac_abs_g_gt_5']:.4f} half_span={max_half_span:.1f} "
          f"(<88: {st['btl_limit_88']})")

all_span_ok = all(v["btl_limit_88"] for v in gate_stats.values())
R["gate_audit"] = {"per_layer": gate_stats, "all_btl_valid": all_span_ok}
print(f"GATE 2.3: BTL half-span valid on all layers: {all_span_ok}", flush=True)
with open("/content/results/gate_audit.json", "w") as fh:
    json.dump(R["gate_audit"], fh, indent=2)

In [ ]:
#@title 4. [RUN 2] layer parity: our jax conventions + chunk kernel vs captured NVlabs I/O
# Conventions (validated by tests/test_inference_v03.py::test_layer_conventions_vs_serial):
# causal depthwise conv k=4 + silu; q/k l2norm; sigmoid b/w; checkpoint gate
# construction g = -exp(A_log) * softplus(f_proj(x) + dt_bias); chunk kernel
# (bt=128, H=16, D=128 — the REAL checkpoint geometry); RMSNormSwishGate
# epilogue; o_proj. Compared against the NVlabs captured mixer I/O.
import jax, jax.numpy as jnp, numpy as np
from atomic_gdn2.config import make_cfg
from atomic_gdn2.inference import chunk_forward_inference

cfg_p = make_cfg(bt=128, bc=64, mb=8, score_bs=128, bs2=32,
                 interpret=not any("tpu" in str(d).lower() for d in jax.devices()),
                 g_max=160.0 / 32)
scale = 1.0

def our_mixer(x_bld, Wl):
    """x_bld: (1, L, H, Dh) head-major fp32 (post-norm_1). Returns (1, L, D)."""
    x = x_bld.transpose(0, 2, 1, 3).reshape(x_bld.shape[0], x_bld.shape[1], -1)  # (1, L, D)
    g_ = lambda suf: jnp.asarray(Wl[suf])
    Wq, Wk, Wv = g_("q_proj.weight"), g_("k_proj.weight"), g_("v_proj.weight")
    Ww, Wb = g_("w_proj.weight"), g_("b_proj.weight")
    Wf0, Wf1 = g_("f_proj.0.weight"), g_("f_proj.1.weight")
    Wg0, Wg1, Wg1b = g_("g_proj.0.weight"), g_("g_proj.1.weight"), g_("g_proj.1.bias")
    A_log, dt_bias = g_("A_log"), g_("dt_bias")
    Wo, Wnorm = g_("o_proj.weight"), g_("o_norm.weight")
    Cq, Ck, Cv = g_("q_conv1d.conv.weight"), g_("k_conv1d.conv.weight"), g_("v_conv1d.conv.weight")

    def conv(x_t, wc):
        # x_t: (1, D, L); wc: (D, 4) -> causal depthwise, left-pad 3
        xp = jnp.pad(x_t, ((0, 0), (0, 0), (3, 0)))
        y = sum(xp[:, :, kk:kk + x_t.shape[-1]] * wc[:, kk][:, None] for kk in range(4))
        return y                                   # (1, D, L)
    def l2(t):
        return t * jax.lax.rsqrt((t * t).sum(-1, keepdims=True) + 1e-6)

    q = jax.nn.silu(conv(x @ Wq.T, Cq))            # (1, D, L)
    k = jax.nn.silu(conv(x @ Wk.T, Ck))
    v = jax.nn.silu(conv(x @ Wv.T, Cv))
    # to head-major (1, H, L, Dh)
    def heads(t, dh):
        return t.reshape(1, t.shape[1], -1, dh).transpose(0, 2, 1, 3)
    q, k = heads(q, 128), heads(k, 128)
    v = heads(v, 128)
    b = jax.nn.sigmoid(x @ Wb.T).reshape(1, x.shape[1], H, 128).transpose(0, 2, 1, 3)
    w = jax.nn.sigmoid(x @ Ww.T).reshape(1, x.shape[1], H, 128).transpose(0, 2, 1, 3)
    f = x @ Wf0.T @ Wf1.T                          # (1, L, K)
    g = (-jnp.exp(A_log).reshape(1, H, 1)
         * jax.nn.softplus(f.reshape(1, f.shape[1], H, 128)
         + dt_bias.reshape(1, 1, Dk))).reshape(1, f.shape[1], H, 128).transpose(0, 2, 1, 3)

    o, hf = chunk_forward_inference(q, k, v, w, b, g, scale, cfg_p)
    # epilogue: RMSNormSwishGate(o, gate) then o_proj
    gate = heads(x @ Wg0.T @ Wg1.T + Wg1b, 128)    # (1, H, L, Dh)
    o3 = o[0].transpose(1, 0, 2)                   # (L, H, Dh)
    v_rms = jnp.sqrt((o3 ** 2).mean(-1, keepdims=True) + 1e-5)
    o3 = o3 / (v_rms + 1e-5) * Wnorm * jax.nn.silu(gate[0].transpose(1, 0, 2))
    o = o3.reshape(1, o3.shape[0], -1) @ Wo.T
    return o                                       # (1, L, D)

print("parity harness ready")

In [ ]:
#@title 5. [RUN 2] parity over layers (rel L2 <= 1e-2 gate)
H = 16; Dh = 128
def W_layer(i):
    return {k2.split("linear_attn.")[-1]: v for k2, v in sd.items()
            if k2.startswith(f"model.layers.{i}.linear_attn.")}
results = {}
for i in LAYERS:
    x_in = captured[i]["x"]                        # (1, L, D) torch
    x_bld = jnp.asarray(np.asarray(x_in).reshape(1, x_in.shape[1], H, Dh)
                        .transpose(0, 2, 1, 3))
    o_nv = captured[i]["o"]                        # (1, L, D) torch
    o_our = our_mixer(x_bld, W_layer(i), cfg_p)
    rel = float(np.linalg.norm(np.asarray(o_our[0]) - np.asarray(o_nv[0]))
                / (np.linalg.norm(np.asarray(o_nv[0])) + 1e-9))
    results[i] = rel
    print(f"layer {i:2d}: rel L2 = {rel:.5f}")

rel_max = max(results.values())
R["parity"] = {"per_layer": results, "rel_l2_max": rel_max,
               "gate": rel_max <= 1e-2}
with open("/content/results/parity.json", "w") as fh:
    json.dump(R["parity"], fh, indent=2)
print(f"GATE 2.4 layer parity: rel L2 max = {rel_max:.5f} -> "
      f"{'PASS' if rel_max <= 1e-2 else 'FAIL'} (contract 1e-2)", flush=True)

In [ ]:
#@title 6. [RUN 3] 0.3.1 kernel gates on TPU (new signatures, checkpoint gates)
import traceback
from atomic_gdn2.config import make_cfg
from atomic_gdn2.inference import decode_step, chunk_forward_inference
from atomic_gdn2.reference import token_serial_ref

G = R["gates"]
cfg3 = make_cfg(bt=64, bc=32, mb=8, score_bs=64, bs2=16,
                interpret=not is_tpu, g_max=160.0 / 16)
key = jax.random.PRNGKey(41)
B, H, D, L, scale = 2, 16, 128, 128, 1.0
q = jax.random.normal(key, (B, L, H, D)) * 0.5
k = jax.random.normal(key+1, (B, L, H, D)) * 0.5
v = jax.random.normal(key+2, (B, L, H, D))
w = jax.nn.sigmoid(jax.random.normal(key+3, (B, L, H, D)))
b = jax.nn.sigmoid(jax.random.normal(key+4, (B, L, H, D)))
g = -jnp.exp(jax.random.normal(key+5, (B, L, H, D)) * 0.39 - 1.0)
hm = lambda t: jnp.transpose(t, (0, 2, 1, 3))

def check(name, fn):
    try:
        fn()
        G[name] = {"ok": True}
        print(f"[RUN 3] {name}: PASS")
    except Exception as e:
        G[name] = {"ok": False, "err": repr(e)[:250]}
        print(f"[RUN 3] {name}: FAIL {repr(e)[:180]}")
        traceback.print_exc()

def _r3_fused_decode_with_checkpoint_gates():
    blk = model.transformer.h[0].attn
    a_log = np.asarray(blk.A_log.float().numpy())
    dt = np.asarray(blk.dt_bias.float().numpy())
    g_raw = jax.random.normal(key+9, (B, H, D)) * 2.0
    g_act = -np.exp(a_log)[None, :, None] * np.log1p(np.exp(np.asarray(g_raw) + dt[None, None, :]))
    state0 = jnp.zeros((B, H, D, D), jnp.float32)
    qn = q[:, :, 0] / np.sqrt((q[:, :, 0]**2).sum(-1, keepdims=True) + 1e-6)
    kn = k[:, :, 0] / np.sqrt((k[:, :, 0]**2).sum(-1, keepdims=True) + 1e-6)
    o_pre, s_pre = decode_step(qn, kn, v[:, :, 0], w[:, :, 0], b[:, :, 0],
                               jnp.asarray(g_act), state0, scale, cfg=cfg3)
    o_fus, s_fus = decode_step(q[:, :, 0], k[:, :, 0], v[:, :, 0],
                               w[:, :, 0], b[:, :, 0], g_raw, state0, scale,
                               fuse_prologue=True, a_log=jnp.asarray(a_log),
                               dt_bias=jnp.asarray(dt), cfg=cfg3)
    np.testing.assert_allclose(np.asarray(o_pre), np.asarray(o_fus), rtol=1e-3, atol=1e-3)
check("R3_fused_decode_checkpoint_gates", _r3_fused_decode_with_checkpoint_gates)

def _r3_a1_batch_rows_independent():
    seg = jnp.zeros((B, L), jnp.int32)
    seg = seg.at[1, 64:].set(1)
    o_pack, _ = chunk_forward_inference(hm(q), hm(k), hm(v), hm(w), hm(b), hm(g),
                                        scale, cfg3, segment_ids=seg)
    o_row1, _ = chunk_forward_inference(hm(q[1:2]), hm(k[1:2]), hm(v[1:2]),
                                        hm(w[1:2]), hm(b[1:2]), hm(g[1:2]),
                                        scale, cfg3, segment_ids=seg[1:2])
    d = np.abs(np.asarray(o_pack)[1] - np.asarray(o_row1)[0]).max()
    assert d < 5e-2, f"row 1 contaminated: {d}"
check("R3_a1_batch_rows_independent", _r3_a1_batch_rows_independent)

def _r3_a2_mid_chunk_raises():
    seg = jnp.cumsum((jnp.arange(L) == 70).astype(jnp.int32)[None], axis=1)
    try:
        chunk_forward_inference(hm(q), hm(k), hm(v), hm(w), hm(b), hm(g),
                                scale, cfg3, segment_ids=seg)
        raise AssertionError("mid-chunk boundary did not raise")
    except ValueError as e:
        assert "boundary inside chunk" in str(e)
check("R3_a2_mid_chunk_raises", _r3_a2_mid_chunk_raises)

def _r3_a6_domain_raises():
    g_bad = -jnp.full((B, L, H, D), 50.0)   # |g| > g_max=10 for bs2=16
    try:
        chunk_forward_inference(hm(q), hm(k), hm(v), hm(w), hm(b), hm(g_bad),
                                scale, cfg3)
        raise AssertionError("out-of-domain g did not raise")
    except ValueError as e:
        assert "outside the layer domain" in str(e)
check("R3_a6_domain_raises", _r3_a6_domain_raises)

def _r3_compiled_prefill_vs_serial():
    o_ref, hf_ref = token_serial_ref(q, k, v, g, b, w, scale)
    o, hf = chunk_forward_inference(hm(q), hm(k), hm(v), hm(w), hm(b), hm(g),
                                    scale, cfg3)
    np.testing.assert_allclose(np.asarray(hm(o)), np.asarray(o_ref),
                               rtol=0.15, atol=0.15)
check("R3_compiled_prefill_vs_serial", _r3_compiled_prefill_vs_serial)

print("RUN 3 gates:", {k: v.get("ok") for k, v in G.items() if k.startswith("R3")},
      flush=True)
with open("/content/results/run3_gates.json", "w") as fh:
    json.dump({k: v for k, v in G.items() if k.startswith("R3")}, fh, indent=2,
              default=str)

In [ ]:
#@title 7. EXPORT: results + zip
import json, subprocess
summary = {"all_ok": all(v.get("ok", True) for k, v in R["gates"].items()
                         if isinstance(v, dict)),
           "gates": {k: (v.get("ok") if isinstance(v, dict) else v)
                     for k, v in R["gates"].items()},
           "parity": R.get("parity"), "gate_audit": R.get("gate_audit"),
           "env": R["env"]}
with open("/content/results/REPORT.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print("GDN2_V031_REPORT: " + json.dumps(summary))
subprocess.run("cd /content && zip -q -r gdn2_v031_stage2.zip results",
               shell=True)
print("zip: /content/gdn2_v031_stage2.zip")